# ANVESH: 12-Factor Machine Learning Evaluation & Forensic Diagnostics
## SIH 2026 Problem Statement: SIH26106 — Email Threat Intelligence Platform

This notebook renders all **12 essential diagnostic figures** demanded by technical evaluators and academic judges:
1. **Class Distribution** (Corpus Partition Breakdown)
2. **Correlation Heatmap** (Top Semantic & Syntactic Token Correlations)
3. **Learning Curve** (Training vs Validation Convergence)
4. **Confusion Matrix Heatmap** (Classification Admissibility)
5. **ROC-AUC Curve** (Sensitivity vs Specificity Tradeoff)
6. **Precision-Recall Curve** (High-Assurance Imbalanced Coverage)
7. **Precision / Recall / F1 vs Threshold** (Optimal Operational Cutoff)
8. **Feature Importance** (Log-Odds Linguistic Drivers)
9. **SHAP Summary / Beeswarm Plot** (Transparent Evidence Attribution)
10. **Error & Confidence Margin Analysis** (Residual Uncertainty Analysis)
11. **Probability Calibration Curve** (Reliability Diagram & Brier Loss)
12. **Multi-Model Benchmark Comparison** (LR vs Naive Bayes vs SVM vs Random Forest)

In [ ]:
# 1. Install & Import Dependencies
!pip install --quiet scikit-learn joblib pandas numpy matplotlib seaborn shap

import os, sys, json, html, re, unicodedata, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import learning_curve
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score, accuracy_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
import shap

# Modern dark cybersecurity styling
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 140

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Environment initialized successfully with scikit-learn & SHAP!')

In [ ]:
# 2. Load Training and Validation Corpora
def load_jsonl(filepath):
    records = []
    if not os.path.exists(filepath):
        print(f'Please upload {filepath} using the upload button below:')
        from google.colab import files
        uploaded = files.upload()
        filepath = list(uploaded.keys())[0]
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_raw = load_jsonl('train.jsonl')
val_raw = load_jsonl('val.jsonl')

print(f'Total Raw Train: {len(train_raw):,} | Total Raw Val: {len(val_raw):,}')

In [ ]:
# 3. Text Normalization Pipeline (RFC 5322 Invariant)
URL_PATTERN = re.compile(r"""(?:https?://|www\.)[^\s<>"'{}|\\^`]+""", re.IGNORECASE)
EMAIL_PATTERN = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
CURRENCY_PATTERN = re.compile(r'[\$£€₹¥]\s*\d+(?:[.,]\d+)*(?:\s*(?:million|billion|thousand|k|m|usd|inr|eur|gbp))?', re.IGNORECASE)
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')
MIME_ARTIFACT_PATTERN = re.compile(r'(?:--=_[A-Za-z0-9._=-]+|Content-Type:[^\n]+|charset=[^\n]+|Content-Transfer-Encoding:[^\n]+)', re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r'\s+')

def clean_email_text(text):
    if not text or not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFKC', text)
    text = html.unescape(text)
    text = MIME_ARTIFACT_PATTERN.sub(' ', text)
    text = HTML_TAG_PATTERN.sub(' ', text)
    text = URL_PATTERN.sub(' __URL_TOKEN__ ', text)
    text = EMAIL_PATTERN.sub(' __EMAIL_TOKEN__ ', text)
    text = CURRENCY_PATTERN.sub(' __CURRENCY_TOKEN__ ', text)
    return WHITESPACE_PATTERN.sub(' ', text).strip()

def normalize_pair(subject, body):
    s = clean_email_text(subject)
    b = clean_email_text(body)
    return f'{s} {b}' if (s and b) else (s or b or 'empty_email')

# Prepare X and y
train_binary = [r for r in train_raw if r.get('anvesh_label') in ('BENIGN', 'THREAT_PHISHING')]
val_binary = [r for r in val_raw if r.get('anvesh_label') in ('BENIGN', 'THREAT_PHISHING')]

X_train = [normalize_pair(r.get('subject', ''), r.get('body', '')) for r in train_binary]
y_train = np.array([1 if r['anvesh_label'] == 'THREAT_PHISHING' else 0 for r in train_binary], dtype=int)

X_val = [normalize_pair(r.get('subject', ''), r.get('body', '')) for r in val_binary]
y_val = np.array([1 if r['anvesh_label'] == 'THREAT_PHISHING' else 0 for r in val_binary], dtype=int)

print(f'Train Samples: {len(X_train):,} (Phishing: {sum(y_train):,}, Benign: {len(y_train)-sum(y_train):,})')
print(f'Val Samples:   {len(X_val):,} (Phishing: {sum(y_val):,}, Benign: {len(y_val)-sum(y_val):,})')

In [ ]:
# 4. Fit Primary Model 1 Pipeline
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=12000,
    sublinear_tf=True,
    token_pattern=r"(?u)\b\w+\b|__\w+__|[$]\d+"
)
clf = LogisticRegression(C=1.5, class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)
pipeline = Pipeline([('tfidf', tfidf), ('clf', clf)])

print('Fitting TF-IDF + Balanced Logistic Regression...')
pipeline.fit(X_train, y_train)
y_val_probs = pipeline.predict_proba(X_val)[:, 1]
y_val_preds = (y_val_probs >= 0.5).astype(int)
print('Training complete and validation predictions generated!')

In [ ]:
# ====================================================================
# GRAPH 1: Class Distribution & Partition Breakdown
# ====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

splits = ['Training Set (70%)', 'Validation Set (30%)']
benign_counts = [len(y_train) - sum(y_train), len(y_val) - sum(y_val)]
phish_counts = [sum(y_train), sum(y_val)]

x = np.arange(len(splits))
width = 0.35

rects1 = ax1.bar(x - width/2, benign_counts, width, label='Benign (0)', color='#3b82f6')
rects2 = ax1.bar(x + width/2, phish_counts, width, label='Phishing Threat (1)', color='#ef4444')

ax1.set_ylabel('Sample Count', fontsize=11, weight='bold')
ax1.set_title('Graph 1A: Absolute Class Breakdown per Partition', fontsize=12, weight='bold', pad=12)
ax1.set_xticks(x)
ax1.set_xticklabels(splits, fontsize=10, weight='semibold')
ax1.legend(facecolor='#1e293b', edgecolor='#334155')
ax1.grid(axis='y', linestyle=':', color='#334155')
ax1.bar_label(rects1, padding=3, color='#e2e8f0', weight='bold')
ax1.bar_label(rects2, padding=3, color='#e2e8f0', weight='bold')

# 1B: Proportion Pie
ax2.pie([sum(y_train) + sum(y_val), (len(y_train)-sum(y_train)) + (len(y_val)-sum(y_val))], 
        labels=['Phishing Threats', 'Benign Corporate'],
        colors=['#ef4444', '#3b82f6'],
        autopct='%1.1f%%',
        startangle=140,
        wedgeprops={'edgecolor': '#0f172a', 'linewidth': 2},
        textprops={'color': '#f8fafc', 'weight': 'bold'})
ax2.set_title('Graph 1B: Total Governed Corpus Ratio', fontsize=12, weight='bold', pad=12)

plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 2: Feature Correlation Heatmap (Top N-Grams)
# ====================================================================
# Extract top 15 most influential tokens
vec = pipeline.named_steps['tfidf']
lr = pipeline.named_steps['clf']
feature_names = np.array(vec.get_feature_names_out())
coefs = lr.coef_[0]
top_indices = np.argsort(np.abs(coefs))[-14:][::-1]
top_tokens = feature_names[top_indices]

# Transform validation subset to get token co-occurrences
X_val_vec = vec.transform(X_val[:500])[:, top_indices].toarray()
df_features = pd.DataFrame(X_val_vec, columns=top_tokens)
corr_matrix = df_features.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-0.3, vmax=1.0, annot=True, fmt='.2f',
            linewidths=0.5, linecolor='#0f172a', cbar_kws={'label': 'Pearson Correlation'})
plt.title('Graph 2: Inter-Token Semantic Correlation Heatmap', fontsize=12, weight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9, weight='semibold')
plt.yticks(fontsize=9, weight='semibold')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 3: Learning Curve (Train vs Validation Convergence)
# ====================================================================
print('Computing Learning Curve across training set sizes (this takes ~10s)...')
train_sizes, train_scores, val_scores = learning_curve(
    pipeline,
    X_train,
    y_train,
    cv=3,
    train_sizes=np.linspace(0.15, 1.0, 5),
    scoring='accuracy',
    n_jobs=-1,
    random_state=RANDOM_SEED
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, 'o-', color='#3b82f6', lw=2.5, label='Training Accuracy')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#3b82f6')

plt.plot(train_sizes, val_mean, 's-', color='#10b981', lw=2.5, label='Cross-Validation Accuracy')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#10b981')

plt.title('Graph 3: Model 1 Learning Curve — Convergence & Generalization', fontsize=12, weight='bold', pad=15)
plt.xlabel('Number of Training Samples', fontsize=11, weight='semibold')
plt.ylabel('Classification Accuracy', fontsize=11, weight='semibold')
plt.ylim([0.90, 1.01])
plt.grid(linestyle=':', color='#334155')
plt.legend(loc='lower right', facecolor='#1e293b', edgecolor='#334155')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 4: Confusion Matrix Heatmap
# ====================================================================
cm = confusion_matrix(y_val, y_val_preds)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(7, 5.5))
labels = [
    [f'True Negative (TN)\n{tn:,}\n(Correct Benign)', f'False Positive (FP)\n{fp:,}\n(False Alarm)'],
    [f'False Negative (FN)\n{fn:,}\n(Missed Phish)', f'True Positive (TP)\n{tp:,}\n(Threat Neutralized)']
]

sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=True,
            xticklabels=['Predicted Benign (0)', 'Predicted Phishing (1)'],
            yticklabels=['Actual Benign (0)', 'Actual Phishing (1)'],
            annot_kws={'size': 10, 'weight': 'bold', 'color': '#ffffff'},
            linewidths=2, linecolor='#0f172a')

plt.title(f'Graph 4: Confusion Matrix\nAccuracy: {(tp+tn)/(tp+tn+fp+fn):.2%} | FPR: {fp/(fp+tn):.2%}', fontsize=12, weight='bold', pad=15)
plt.xlabel('Predicted Security Verdict', fontsize=11, weight='semibold')
plt.ylabel('Ground Truth Label', fontsize=11, weight='semibold')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 5 & 6: ROC-AUC Curve and Precision-Recall Curve
# ====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 5: ROC Curve
fpr, tpr, _ = roc_curve(y_val, y_val_probs)
roc_auc = roc_auc_score(y_val, y_val_probs)
ax1.plot(fpr, tpr, color='#06b6d4', lw=2.5, label=f'ANVESH Model 1 (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], color='#64748b', lw=1.5, linestyle='--', label='Random Baseline')
ax1.set_xlim([-0.02, 1.0])
ax1.set_ylim([0.0, 1.03])
ax1.set_title('Graph 5: Receiver Operating Characteristic (ROC)', fontsize=12, weight='bold', pad=12)
ax1.set_xlabel('False Positive Rate (FPR)', fontsize=10, weight='semibold')
ax1.set_ylabel('True Positive Rate (TPR / Recall)', fontsize=10, weight='semibold')
ax1.grid(linestyle=':', color='#334155')
ax1.legend(loc='lower right', facecolor='#1e293b', edgecolor='#334155')

# 6: Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_val, y_val_probs)
ap = average_precision_score(y_val, y_val_probs)
ax2.plot(rec, prec, color='#10b981', lw=2.5, label=f'Model 1 (AP = {ap:.4f})')
ax2.axhline(y=sum(y_val)/len(y_val), color='#f59e0b', lw=1.5, linestyle='--', label=f'Prevalence ({sum(y_val)/len(y_val):.2%})')
ax2.set_xlim([0.0, 1.02])
ax2.set_ylim([0.0, 1.03])
ax2.set_title('Graph 6: Precision-Recall Tradeoff Curve', fontsize=12, weight='bold', pad=12)
ax2.set_xlabel('Recall (Detection Coverage)', fontsize=10, weight='semibold')
ax2.set_ylabel('Precision (Forensic Admissibility)', fontsize=10, weight='semibold')
ax2.grid(linestyle=':', color='#334155')
ax2.legend(loc='lower left', facecolor='#1e293b', edgecolor='#334155')

plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 7: Precision, Recall & F1 Across Decision Thresholds
# ====================================================================
thresholds = np.linspace(0.05, 0.95, 30)
precisions = []
recalls = []
f1_scores = []

for th in thresholds:
    preds_th = (y_val_probs >= th).astype(int)
    precisions.append(precision_score(y_val, preds_th, zero_division=0))
    recalls.append(recall_score(y_val, preds_th, zero_division=0))
    f1_scores.append(f1_score(y_val, preds_th, zero_division=0))

plt.figure(figsize=(9, 5))
plt.plot(thresholds, precisions, 'b-', lw=2, label='Precision (Zero False Positives)')
plt.plot(thresholds, recalls, 'r-', lw=2, label='Recall (Threat Coverage)')
plt.plot(thresholds, f1_scores, 'g--', lw=2.5, label='F1-Score (Harmonic Mean)')
plt.axvline(x=0.5, color='#f59e0b', linestyle=':', lw=2, label='ANVESH Operating Cutoff (0.50)')

plt.title('Graph 7: Forensic Metric Stability Across Classification Thresholds', fontsize=12, weight='bold', pad=15)
plt.xlabel('Decision Cutoff Probability Threshold', fontsize=11, weight='semibold')
plt.ylabel('Metric Score [0.0 - 1.0]', fontsize=11, weight='semibold')
plt.ylim([0.80, 1.02])
plt.grid(linestyle=':', color='#334155')
plt.legend(loc='lower center', facecolor='#1e293b', edgecolor='#334155')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 8: Feature Importance (Logistic Regression Log-Odds)
# ====================================================================
top_pos_idx = np.argsort(coefs)[-10:]
top_neg_idx = np.argsort(coefs)[:10]
combined = np.concatenate([top_neg_idx, top_pos_idx])

feat_names_sub = feature_names[combined]
feat_weights = coefs[combined]
bar_colors = ['#3b82f6' if w < 0 else '#ef4444' for w in feat_weights]

plt.figure(figsize=(8.5, 6))
y_pos = np.arange(len(feat_names_sub))
bars = plt.barh(y_pos, feat_weights, color=bar_colors, height=0.65)

plt.yticks(y_pos, feat_names_sub, fontsize=10, weight='semibold')
plt.xlabel('Log-Odds Coefficient Weight', fontsize=11, weight='semibold')
plt.title('Graph 8: Top 20 Linguistic Indicators (Red: Phishing | Blue: Benign)', fontsize=12, weight='bold', pad=15)
plt.axvline(x=0, color='#64748b', lw=1)
plt.grid(axis='x', linestyle=':', color='#334155')

for bar in bars:
    val = bar.get_width()
    offset = 0.08 if val >= 0 else -0.08
    align = 'left' if val >= 0 else 'right'
    plt.text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:+.2f}',
             va='center', ha=align, fontsize=9, weight='bold', color='#f8fafc')

plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 9: SHAP Explainability (Shapley Value Feature Attributions)
# ====================================================================
print('Calculating SHAP values for model explainability (Section 63 BSA 2023 compliance)...')

# Sample 300 instances for fast SHAP calculation
sample_texts = X_val[:300]
X_sample_vec = vec.transform(sample_texts)

# Masker & LinearExplainer
masker = shap.maskers.Independent(data=X_sample_vec[:100].toarray())
explainer = shap.LinearExplainer(lr, masker=masker)
shap_values = explainer(X_sample_vec.toarray())

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values.values, X_sample_vec.toarray(), feature_names=feature_names, 
                  max_display=15, show=False)
plt.title('Graph 9: SHAP Beeswarm Summary Plot — Evidence Attribution', fontsize=12, weight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 10: Error & Confidence Margin Residual Analysis
# ====================================================================
margins = np.abs(y_val_probs - 0.5)
correct_mask = (y_val == y_val_preds)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

# 10A: Margin Density
ax1.hist(margins[correct_mask], bins=25, color='#10b981', alpha=0.8, label=f'Correct Decisions ({sum(correct_mask):,})')
if sum(~correct_mask) > 0:
    ax1.hist(margins[~correct_mask], bins=10, color='#ef4444', alpha=0.9, label=f'Misclassifications ({sum(~correct_mask)})')
ax1.set_title('Graph 10A: Confidence Margin from Decision Boundary', fontsize=11, weight='bold', pad=12)
ax1.set_xlabel('Distance from 0.50 Decision Cutoff (|P - 0.5|)', fontsize=10, weight='semibold')
ax1.set_ylabel('Sample Frequency', fontsize=10, weight='semibold')
ax1.grid(linestyle=':', color='#334155')
ax1.legend(facecolor='#1e293b', edgecolor='#334155')

# 10B: Probability Spread Scatter
sample_indices = np.random.choice(len(y_val), size=min(400, len(y_val)), replace=False)
colors = ['#ef4444' if y == 1 else '#3b82f6' for y in y_val[sample_indices]]
ax2.scatter(sample_indices, y_val_probs[sample_indices], c=colors, alpha=0.65, s=25)
ax2.axhline(0.5, color='#f59e0b', linestyle='--', lw=1.5, label='Cutoff 0.5')
ax2.set_title('Graph 10B: Prediction Separation Scatter', fontsize=11, weight='bold', pad=12)
ax2.set_xlabel('Sample Index (Subset)', fontsize=10, weight='semibold')
ax2.set_ylabel('Phishing Probability', fontsize=10, weight='semibold')
ax2.grid(linestyle=':', color='#334155')
ax2.legend(facecolor='#1e293b', edgecolor='#334155')

plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 11: Calibration Curve (Reliability Diagram & Brier Loss)
# ====================================================================
prob_true, prob_pred = calibration_curve(y_val, y_val_probs, n_bins=10)
brier_loss = brier_score_loss(y_val, y_val_probs)

plt.figure(figsize=(7, 5))
plt.plot(prob_pred, prob_true, 's-', color='#06b6d4', lw=2.5, label=f'Model 1 (Brier Score = {brier_loss:.4f})')
plt.plot([0, 1], [0, 1], 'k--', color='#64748b', lw=1.5, label='Perfect Calibration')

plt.title(f'Graph 11: Reliability Diagram (Probability Calibration)\nBrier Score: {brier_loss:.4f} (Optimal $\\to$ 0.000)', fontsize=12, weight='bold', pad=15)
plt.xlabel('Mean Predicted Phishing Probability', fontsize=11, weight='semibold')
plt.ylabel('Empirical Fraction of Positives', fontsize=11, weight='semibold')
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.grid(linestyle=':', color='#334155')
plt.legend(loc='lower right', facecolor='#1e293b', edgecolor='#334155')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# GRAPH 12: Multi-Model Benchmark Comparison
# ====================================================================
print('Benchmarking 4 Candidate Classifier Architectures...')
models = {
    'Logistic Regression (ANVESH)': LogisticRegression(C=1.5, class_weight='balanced', max_iter=1000, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM (SGD)': SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    'Random Forest (Trees=50)': RandomForestClassifier(n_estimators=50, max_depth=15, random_state=42)
}

X_train_vec = vec.transform(X_train)
X_val_vec = vec.transform(X_val)

comparison_results = []
for name, m in models.items():
    m.fit(X_train_vec, y_train)
    preds = m.predict(X_val_vec)
    acc = accuracy_score(y_val, preds)
    prec = precision_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)
    comparison_results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

df_compare = pd.DataFrame(comparison_results)

plt.figure(figsize=(11, 5.5))
x = np.arange(len(df_compare))
width = 0.2

plt.bar(x - 1.5*width, df_compare['Accuracy'], width, label='Accuracy', color='#3b82f6')
plt.bar(x - 0.5*width, df_compare['Precision'], width, label='Precision', color='#06b6d4')
plt.bar(x + 0.5*width, df_compare['Recall'], width, label='Recall', color='#10b981')
plt.bar(x + 1.5*width, df_compare['F1-Score'], width, label='F1-Score', color='#f59e0b')

plt.ylabel('Validation Score', fontsize=11, weight='bold')
plt.title('Graph 12: Architectural Model Comparison Benchmark (SIH 2026)', fontsize=12, weight='bold', pad=15)
plt.xticks(x, df_compare['Model'], fontsize=9.5, weight='semibold', rotation=15, ha='right')
plt.ylim([0.85, 1.02])
plt.grid(axis='y', linestyle=':', color='#334155')
plt.legend(loc='lower left', facecolor='#1e293b', edgecolor='#334155')
plt.tight_layout()
plt.show()

print('\nModel Comparison Table:')
print(df_compare.to_string(index=False))